In [1]:
# Author: Nathan Miller and Gergely Zahoranszky-Kohalmi, PhD
#
# Organization: National Center for Advancing Translational Sciences
#

In [2]:
# Env: syngps_rev
import pandas as pd
import os

from rdkit import Chem
from rdkit.Chem import AllChem
#from pandarallel import pandarallel

# Hide InsecureRequestWarning
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

import requests
import json

In [3]:

FNAME_IN = '../data/output/tms.tsv'

SEARCH_DEPTH = 5

SEARCH_TYPE = 'shortest_path'


INCLUDE_AVAILABILITY_INFO = False

ANNOTATE_REACTIONS = False

GRAPH_BACKEND = 'memgraph'

INVENTORY_SOURCE = 'askcos'


INCLUDE_EMOLECULES_INV = False
INCLUDE_RETROSTAR_INV = True
INCLUDE_SIMPRETRO_INV = True

SYNTH_GRAPH_API = 'http://localhost:8002/syngps-app/api/v1/prediction/fetch_synthesis_graph/'

# LASM: Leaves As Starting Materials
JSON_FOLDER_LASM = '../data/output/synthesis_graphs_lasm'

JSON_FOLDER_INV= '../data/output/synthesis_graphs_inv'

SYNTH_GRAPH_REQUEST_TYPE = 'POST'

In [4]:
def generate_synth_graph_payload_lasm (target_inchikey):
    payload = {
        "target_molecule_inchikey": target_inchikey,
        "reaction_steps": SEARCH_DEPTH,
        "query_type": SEARCH_TYPE,
        "leaves_as_sm": True,
        "include_availability_info": INCLUDE_AVAILABILITY_INFO,
        "annotate_reactions": ANNOTATE_REACTIONS,
        "graph_backend": GRAPH_BACKEND
    }

    return (payload)



def generate_synth_graph_payload_inv (target_inchikey):
    payload = {
        "target_molecule_inchikey": target_inchikey,
        "reaction_steps": SEARCH_DEPTH,
        "query_type": SEARCH_TYPE,
        "leaves_as_sm": False,
        "include_availability_info": INCLUDE_AVAILABILITY_INFO,
        "annotate_reactions": ANNOTATE_REACTIONS,
        "graph_backend": GRAPH_BACKEND,
        "inventory_source": INVENTORY_SOURCE
    }

    return (payload)



def process_inchikey_lasm(row, idx, total):
    inchikey = row['inchikey']
    payload = generate_synth_graph_payload_lasm(inchikey)

    print (payload)
    try:
        response = requests.post(SYNTH_GRAPH_API,
                                 json=payload,
                                 headers={'Content-Type': 'application/json',
                                        'accept': 'application/json'},
                                 verify=False)
        

        progress_msg = f"[{idx+1}/{total}] Processed {inchikey} - Status: {response.status_code}"
        print(progress_msg)
        if response.status_code == 200:
            response_filename = f"{JSON_FOLDER_LASM}/{inchikey}_response.json"
            with open(response_filename, 'w') as response_file:
                json.dump(response.json(), response_file, indent=4)
            return (inchikey, True)
        else:
            # Print error message for failed requests
            try:
                error_msg = response.text
            except Exception:
                error_msg = '<no error message>'
            print(f"Error for {inchikey}: {error_msg}")
            return (inchikey, False)
    except Exception as e:
        print(f"[{idx+1}/{total}] Exception for {inchikey}: {e}")
        return (inchikey, False)

def process_inchikey_inv(row, idx, total):
    inchikey = row['inchikey']
    payload = generate_synth_graph_payload_inv(inchikey)

    print (payload)
    try:
        response = requests.post(SYNTH_GRAPH_API,
                                 json=payload,
                                 headers={'Content-Type': 'application/json',
                                        'accept': 'application/json'},
                                 verify=False)
        

        progress_msg = f"[{idx+1}/{total}] Processed {inchikey} - Status: {response.status_code}"
        print(progress_msg)
        if response.status_code == 200:
            response_filename = f"{JSON_FOLDER_INV}/{inchikey}_response.json"
            with open(response_filename, 'w') as response_file:
                json.dump(response.json(), response_file, indent=4)
            return (inchikey, True)
        else:
            # Print error message for failed requests
            try:
                error_msg = response.text
            except Exception:
                error_msg = '<no error message>'
            print(f"Error for {inchikey}: {error_msg}")
            return (inchikey, False)
    except Exception as e:
        print(f"[{idx+1}/{total}] Exception for {inchikey}: {e}")
        return (inchikey, False)

In [5]:
# Parse input

df = pd.read_csv (FNAME_IN, sep = '\t')

print (df.head)

<bound method NDFrame.head of                         inchikey  \
0    AAZPIQPULVRHOW-SFKJMYEFSA-N   
1    ACMYXPZZKWSEOO-ROUUACIJSA-N   
2    ADROYVWTRVGXHG-UHFFFAOYSA-N   
3    AEMZJBZKICOPPC-DZDWSLRDSA-N   
4    AGOWMJFPYPUNRF-UHFFFAOYSA-N   
..                           ...   
367  ZNTVNNXGPZITBD-UHFFFAOYSA-N   
368  ZQXOMIPSVVBRGV-UHFFFAOYSA-N   
369  ZRBPIAWWRPFDPY-IRXDYDNUSA-N   
370  ZTXVRZLOVRFPEC-UHFFFAOYSA-N   
371  ZVERWTXKKWSSHH-UHFFFAOYSA-N   

                                                smiles  
0    COc1ccc(CO[C@H]2CC[C@@]3(C)C(=CC[C@H]4[C@@H]5C...  
1    CC(C)(C)OC(=O)CC[C@H](NC(=O)OCc1ccccc1)C(=O)N[...  
2                                       COC(CC#N)=NC#N  
3    CC(C)(O)c1ccc(-c2cccc(N(CC34CCC(c5noc(C(C)(C)F...  
4                   Nc1nc(CSc2ccc([N+](=O)[O-])cc2)cs1  
..                                                 ...  
367             COC(=O)c1cn2cc(-c3ccccc3)cc(C(C)C)c2n1  
368           Cn1ccnc1S(=O)Cc1cccc2cc(-c3nccs3)[nH]c12  
369  C=CC(=O)N1CCN(c2nc(

In [6]:

total_inchikeys = df['inchikey'].count()
print(f'Total number of InChIKeys in the DataFrame: {total_inchikeys}')

unique_inchikeys = df['inchikey'].nunique()
print(f'Total number of unique InChIKeys in the DataFrame: {unique_inchikeys}')


Total number of InChIKeys in the DataFrame: 372
Total number of unique InChIKeys in the DataFrame: 372


In [7]:


# Ensure the JSON_FOLDER_LASM exists
if not os.path.exists(JSON_FOLDER_LASM):
    os.makedirs(JSON_FOLDER_LASM)


# Load already processed InChIKeys by checking for existing JSON files
existing_files = set()
for fname in os.listdir(JSON_FOLDER_LASM):
    if fname.endswith('_response.json'):
        inchikey = fname.replace('_response.json', '')
        existing_files.add(inchikey)

# Disable failed_inchikeys checkpointing for speed test
failed_inchikeys = set()

# Filter the DataFrame to only include InChIKeys that have not been processed yet (success or fail)
to_process_df = df[~df['inchikey'].isin(existing_files | failed_inchikeys)]
total_to_process = len(to_process_df)
print(f"Total InChIKeys to process: {total_to_process}")

success_count = 0
failure_count = 0
success_list = []
failure_list = []



# Sequential processing (no threading)
for idx, (_, row) in enumerate(to_process_df.iterrows()):
    inchikey = row['inchikey'].strip()
    result_inchikey, success = process_inchikey_lasm (row, idx, total_to_process)

    print (result_inchikey)
    if success:
        success_count += 1
        success_list.append(result_inchikey)
    else:
        failure_count += 1
        failure_list.append(result_inchikey)

print(f"Total successful responses (LASM): {success_count}")
print(f"Total failed responses (LASM): {failure_count}")

print("Successful InChIKeys (LASM):")
print(success_list)
print("Failed InChIKeys (LASM):")
print(failure_list)

Total InChIKeys to process: 372
{'target_molecule_inchikey': 'AAZPIQPULVRHOW-SFKJMYEFSA-N', 'reaction_steps': 5, 'query_type': 'shortest_path', 'leaves_as_sm': True, 'include_availability_info': False, 'annotate_reactions': False, 'graph_backend': 'memgraph'}
[1/372] Processed AAZPIQPULVRHOW-SFKJMYEFSA-N - Status: 404
Error for AAZPIQPULVRHOW-SFKJMYEFSA-N: {"detail":"Substance not found in synthesis graph: AAZPIQPULVRHOW-SFKJMYEFSA-N"}
AAZPIQPULVRHOW-SFKJMYEFSA-N
{'target_molecule_inchikey': 'ACMYXPZZKWSEOO-ROUUACIJSA-N', 'reaction_steps': 5, 'query_type': 'shortest_path', 'leaves_as_sm': True, 'include_availability_info': False, 'annotate_reactions': False, 'graph_backend': 'memgraph'}
[2/372] Processed ACMYXPZZKWSEOO-ROUUACIJSA-N - Status: 404
Error for ACMYXPZZKWSEOO-ROUUACIJSA-N: {"detail":"Substance not found in synthesis graph: ACMYXPZZKWSEOO-ROUUACIJSA-N"}
ACMYXPZZKWSEOO-ROUUACIJSA-N
{'target_molecule_inchikey': 'ADROYVWTRVGXHG-UHFFFAOYSA-N', 'reaction_steps': 5, 'query_type': '

In [8]:


# Ensure the JSON_FOLDER_INV exists
if not os.path.exists(JSON_FOLDER_INV):
    os.makedirs(JSON_FOLDER_INV)


# Load already processed InChIKeys by checking for existing JSON files
existing_files = set()
for fname in os.listdir(JSON_FOLDER_INV):
    if fname.endswith('_response.json'):
        inchikey = fname.replace('_response.json', '')
        existing_files.add(inchikey)

# Disable failed_inchikeys checkpointing for speed test
failed_inchikeys = set()

# Filter the DataFrame to only include InChIKeys that have not been processed yet (success or fail)
to_process_df = df[~df['inchikey'].isin(existing_files | failed_inchikeys)]
total_to_process = len(to_process_df)
print(f"Total InChIKeys to process: {total_to_process}")

success_count = 0
failure_count = 0
success_list = []
failure_list = []



# Sequential processing (no threading)
for idx, (_, row) in enumerate(to_process_df.iterrows()):
    inchikey = row['inchikey'].strip()
    result_inchikey, success = process_inchikey_inv (row, idx, total_to_process)

    print (result_inchikey)
    if success:
        success_count += 1
        success_list.append(result_inchikey)
    else:
        failure_count += 1
        failure_list.append(result_inchikey)

print(f"Total successful responses (INV): {success_count}")
print(f"Total failed responses (INV): {failure_count}")

print("Successful InChIKeys (INV):")
print(success_list)
print("Failed InChIKeys (INV):")
print(failure_list)

Total InChIKeys to process: 372
{'target_molecule_inchikey': 'AAZPIQPULVRHOW-SFKJMYEFSA-N', 'reaction_steps': 5, 'query_type': 'shortest_path', 'leaves_as_sm': False, 'include_availability_info': False, 'annotate_reactions': False, 'graph_backend': 'memgraph', 'inventory_source': 'askcos'}
[1/372] Processed AAZPIQPULVRHOW-SFKJMYEFSA-N - Status: 404
Error for AAZPIQPULVRHOW-SFKJMYEFSA-N: {"detail":"Substance not found in synthesis graph: AAZPIQPULVRHOW-SFKJMYEFSA-N"}
AAZPIQPULVRHOW-SFKJMYEFSA-N
{'target_molecule_inchikey': 'ACMYXPZZKWSEOO-ROUUACIJSA-N', 'reaction_steps': 5, 'query_type': 'shortest_path', 'leaves_as_sm': False, 'include_availability_info': False, 'annotate_reactions': False, 'graph_backend': 'memgraph', 'inventory_source': 'askcos'}
[2/372] Processed ACMYXPZZKWSEOO-ROUUACIJSA-N - Status: 404
Error for ACMYXPZZKWSEOO-ROUUACIJSA-N: {"detail":"Substance not found in synthesis graph: ACMYXPZZKWSEOO-ROUUACIJSA-N"}
ACMYXPZZKWSEOO-ROUUACIJSA-N
{'target_molecule_inchikey': 'ADRO

In [9]:
print ('[Done.]')

[Done.]
